In [ ]:
import os
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset, load_from_disk, concatenate_datasets, Audio as HFAudio
from transformers import WhisperForConditionalGeneration, WhisperProcessor, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = (device == "cuda" and torch.cuda.is_bf16_supported())
amp_dtype = torch.bfloat16 if use_bf16 else (torch.float16 if device == "cuda" else torch.float32)

print(f"🖥️ Thiết bị hiện tại: {device.upper()} ({torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'}) | Precision: {amp_dtype}")

BASE_DRIVE = "/content/drive/MyDrive/Distil_Whisper_Vi_Yanshee"
STUDENT_INIT_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_init"
CHECKPOINT_DIR = f"{BASE_DRIVE}/checkpoints_large_run"
BEST_MODEL_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_trained"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

# =====================================================================
# 1. TẢI DỮ LIỆU NGƯỜI VIỆT NÓI THẬT (JRHuy/vivos-fleurs)
# =====================================================================
NUM_REAL_SAMPLES = 2000
BATCH_SIZE = 16
EPOCHS = 4

VERIFIED_REPOS = [
    "JRHuy/vivos-fleurs",
    "ademax/vivos-vie-speech2text",
    "quocanh34/viet_vivos",
]

real_train, real_test = None, None
for repo_id in VERIFIED_REPOS:
    try:
        print(f"\n📥 [1/4] Đang tải {NUM_REAL_SAMPLES} mẫu giọng người Việt thật từ '{repo_id}'...")
        real_train = load_dataset(repo_id, split=f"train[:{NUM_REAL_SAMPLES}]")
        real_test = load_dataset(repo_id, split="test[:20]")
        print(f"   ✅ Tải thành công từ '{repo_id}'!")
        break
    except Exception as e:
        print(f"   ⚠️ Thử kho tiếp theo ({e})...")

text_col = "raw_transcription" if "raw_transcription" in real_train.column_names else "transcription"
real_train = real_train.rename_column(text_col, "sentence").cast_column("audio", HFAudio(sampling_rate=16000))
real_test = real_test.rename_column(text_col, "sentence").cast_column("audio", HFAudio(sampling_rate=16000))

keep_cols = ["audio", "sentence"]
real_train = real_train.remove_columns([c for c in real_train.column_names if c not in keep_cols])
real_test = real_test.remove_columns([c for c in real_test.column_names if c not in keep_cols])

# ĐÃ SỬA: Xóa các cột thừa (bao gồm cột 'sentence' cũ) trước khi đổi tên 'whisper_transcript' -> 'sentence'
PSEUDO_YANSHEE_DIR = f"{BASE_DRIVE}/data/yanshee_vi_pseudo_labelled"
if os.path.exists(PSEUDO_YANSHEE_DIR):
    yanshee_ds = load_from_disk(PSEUDO_YANSHEE_DIR)
    y_train = yanshee_ds["train"].remove_columns([c for c in yanshee_ds["train"].column_names if c not in ["audio", "whisper_transcript"]])
    y_test = yanshee_ds["test"].remove_columns([c for c in yanshee_ds["test"].column_names if c not in ["audio", "whisper_transcript"]])
    y_train = y_train.rename_column("whisper_transcript", "sentence").cast_column("audio", HFAudio(sampling_rate=16000))
    y_test = y_test.rename_column("whisper_transcript", "sentence").cast_column("audio", HFAudio(sampling_rate=16000))

    full_train = concatenate_datasets([real_train] + [y_train] * 15).shuffle(seed=42)
    full_test = concatenate_datasets([y_test, real_test.select(range(6))])
else:
    full_train = real_train
    full_test = real_test.select(range(8))

print(f"✅ Tổng dữ liệu huấn luyện: {len(full_train)} mẫu | Tập đánh giá (Test): {len(full_test)} mẫu")

# =====================================================================
# 2. NẠP TEACHER (PhoWhisper-large) & STUDENT (6 Enc / 2 Dec)
# =====================================================================
TEACHER_ID = "vinai/PhoWhisper-large"
print(f"\n🎓 [2/4] Đang nạp Teacher ({TEACHER_ID}) và Student (6 Enc / 2 Dec) lên A100...")

processor = WhisperProcessor.from_pretrained("vinai/PhoWhisper-base")
processor.tokenizer.set_prefix_tokens(language="vi", task="transcribe", predict_timestamps=False)

teacher_model = WhisperForConditionalGeneration.from_pretrained(TEACHER_ID, torch_dtype=amp_dtype).to(device)
teacher_model.eval()

base_m = WhisperForConditionalGeneration.from_pretrained("vinai/PhoWhisper-base")
s_cfg = copy.deepcopy(base_m.config)
s_cfg.decoder_layers = 2
student_model = WhisperForConditionalGeneration(s_cfg)
student_model.load_state_dict(base_m.state_dict(), strict=False)
for s_i, t_i in enumerate([0, 5]):
    student_model.model.decoder.layers[s_i].load_state_dict(base_m.model.decoder.layers[t_i].state_dict())
student_model.generation_config = copy.deepcopy(base_m.generation_config)
del base_m

student_model.to(device)
student_model.freeze_encoder()

# =====================================================================
# 3. COLLATOR CHUẨN WHISPER & VÒNG LẶP DISTILLATION TRÊN A100
# =====================================================================
bos_id = student_model.config.decoder_start_token_id

def collate_fn(batch):
    audios = [item["audio"]["array"] for item in batch]
    input_features = processor(audios, sampling_rate=16000, return_tensors="pt").input_features
    texts = [str(item["sentence"]).lower() if "Yanshee" not in str(item["sentence"]) else str(item["sentence"]) for item in batch]

    labels_batch = processor.tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    labels = labels_batch.input_ids.masked_fill(labels_batch.attention_mask.ne(1), -100)
    if (labels[:, 0] == bos_id).all().cpu().item():
        labels = labels[:, 1:]
    return {"input_features": input_features, "labels": labels, "texts": texts}

train_loader = DataLoader(full_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(full_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

TEMPERATURE = 2.0
def compute_kd_loss(student_logits, teacher_logits, labels):
    vocab_size = student_logits.size(-1)
    loss_pl = F.cross_entropy(student_logits.view(-1, vocab_size), labels.view(-1), ignore_index=-100)
    valid_mask = (labels != -100).unsqueeze(-1)
    s_log_probs = F.log_softmax(student_logits.float() / TEMPERATURE, dim=-1)
    t_probs = F.softmax(teacher_logits.float() / TEMPERATURE, dim=-1)
    kl_token = F.kl_div(s_log_probs, t_probs, reduction="none") * (TEMPERATURE ** 2)
    loss_kl = (kl_token * valid_mask).sum() / valid_mask.sum().clamp(min=1.0)
    return 0.8 * loss_kl + 1.0 * loss_pl, loss_kl.item(), loss_pl.item()

total_steps = len(train_loader) * EPOCHS
optimizer = torch.optim.AdamW([p for p in student_model.parameters() if p.requires_grad], lr=2e-4, betas=(0.9, 0.999))
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=max(10, int(0.08 * total_steps)), num_training_steps=total_steps)

ckpt_path = os.path.join(CHECKPOINT_DIR, "checkpoint_a100_latest.pt")

print(f"\n🚀 [3/4] BẮT ĐẦU HUẤN LUYỆN TRÊN A100 ({EPOCHS} Epochs = {total_steps} Steps)...")
for epoch in range(1, EPOCHS + 1):
    student_model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for step_idx, batch in enumerate(pbar, 1):
        inp = batch["input_features"].to(device)
        lbl = batch["labels"].to(device)

        optimizer.zero_grad()
        with torch.no_grad():
            t_logits = teacher_model(input_features=inp.to(amp_dtype), labels=lbl).logits
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
            s_logits = student_model(input_features=inp, labels=lbl).logits
            loss, l_kl, l_pl = compute_kd_loss(s_logits, t_logits, lbl)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix({"L_KD": f"{loss.item():.2f}", "KL": f"{l_kl:.2f}", "PL": f"{l_pl:.2f}"})

    torch.save({
        "epoch": epoch,
        "model_state_dict": student_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
    }, ckpt_path)
    print(f"   -> Kết thúc Epoch {epoch:02d}/{EPOCHS:02d} | Loss trung bình: {total_loss/len(train_loader):.4f} (Đã lưu Checkpoint)")

student_model.save_pretrained(BEST_MODEL_DIR)
processor.save_pretrained(BEST_MODEL_DIR)
print(f"\n💾 [4/4] Đã lưu Student Model hoàn chỉnh tại: {BEST_MODEL_DIR}")

student_model.eval()
print("\n🎯 KIỂM TRA TRÊN TẬP TEST (CẢ LỆNH YANSHEE LẪN NGƯỜI THẬT VIVOS/FLEURS):")
for batch in test_loader:
    with torch.no_grad():
        gen_ids = student_model.generate(batch["input_features"].to(device), language="vi", task="transcribe", max_new_tokens=50)
        preds = processor.batch_decode(gen_ids, skip_special_tokens=True)
    for gt, pr in zip(batch["texts"], preds):
        print(f"   📌 Gốc    : '{gt}'")
        print(f"   🤖 Student: '{pr}'\n")

🖥️ Thiết bị hiện tại: CUDA (NVIDIA A100-SXM4-40GB) | Precision: torch.bfloat16

📥 [1/4] Đang tải 2000 mẫu giọng người Việt thật từ 'JRHuy/vivos-fleurs'...
   ✅ Tải thành công từ 'JRHuy/vivos-fleurs'!
✅ Tổng dữ liệu huấn luyện: 2120 mẫu | Tập đánh giá (Test): 8 mẫu

🎓 [2/4] Đang nạp Teacher (vinai/PhoWhisper-large) và Student (6 Enc / 2 Dec) lên A100...


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



💾 [4/4] Đã lưu Student Model hoàn chỉnh tại: /content/drive/MyDrive/Distil_Whisper_Vi_Yanshee/models/distil_phowhisper_base_trained

🎯 KIỂM TRA TRÊN TẬP TEST (CẢ LỆNH YANSHEE LẪN NGƯỜI THẬT VIVOS/FLEURS):


[transformers] Both `max_new_tokens` (=50) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


   📌 Gốc    : 'Yanshee ơi tiếng lên phía trước hai bước'
   🤖 Student: 'dân xe ơi tiếng lên phía trước hai bước.'

   📌 Gốc    : 'Yanshee ơi cúi đầu chào khán giả'
   🤖 Student: 'Yanshee ơi cuối đầu chào khán giả.'

   📌 Gốc    : 'nửa vòng trái đất hơn bảy năm'
   🤖 Student: 'nửa vòng trái đất hơn bảy năm.'

   📌 Gốc    : 'bốn chiếc trống đại phèng la'
   🤖 Student: 'bốn chiếc trống đại phèn la.'

   📌 Gốc    : 'của những thế lực hùng mạnh'
   🤖 Student: 'của những thế lực hùng mạnh.'

   📌 Gốc    : 'ông nhặt nó lên'
   🤖 Student: 'ông nhạc nó lên.'

   📌 Gốc    : 'thế nhưng khi giá phôi thép thế giới cao dần'
   🤖 Student: 'thế nhưng khi giá phôi thép thế giới cao dần.'

   📌 Gốc    : 'tên gọi tại địa phương là trắc thối'
   🤖 Student: 'tên gọi tài địa phương là trắc thối.'



In [ ]:
import time
import re
import unicodedata
import torch
import pandas as pd
from datasets import load_dataset, load_from_disk, concatenate_datasets, Audio as HFAudio
from transformers import WhisperForConditionalGeneration, WhisperProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
amp_dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else (torch.float16 if device == "cuda" else torch.float32)

BASE_DRIVE = "/content/drive/MyDrive/Distil_Whisper_Vi_Yanshee"
STUDENT_INIT_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_init"
STUDENT_TRAINED_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_trained"

# 1. TẢI TẬP TEST (50 mẫu người thật VIVOS/FLEURS + Tập Test Yanshee)
print("📥 Đang chuẩn bị tập Test đánh giá (50 câu tiếng Việt thực tế)...")
test_ds = load_dataset("JRHuy/vivos-fleurs", split="test[:48]")
text_col = "raw_transcription" if "raw_transcription" in test_ds.column_names else "transcription"
test_ds = test_ds.rename_column(text_col, "sentence").cast_column("audio", HFAudio(sampling_rate=16000))
test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in ["audio", "sentence"]])

PSEUDO_YANSHEE_DIR = f"{BASE_DRIVE}/data/yanshee_vi_pseudo_labelled"
if os.path.exists(PSEUDO_YANSHEE_DIR):
    y_test = load_from_disk(PSEUDO_YANSHEE_DIR)["test"]
    y_test = y_test.remove_columns([c for c in y_test.column_names if c not in ["audio", "whisper_transcript"]])
    y_test = y_test.rename_column("whisper_transcript", "sentence").cast_column("audio", HFAudio(sampling_rate=16000))
    test_ds = concatenate_datasets([y_test, test_ds])

processor = WhisperProcessor.from_pretrained("vinai/PhoWhisper-base")

# 2. HÀM CHUẨN HÓA & TÍNH CHI TIẾT S, D, I, WER, CER (Từ Chặng 3 & 6)
NUM_TO_VI = {"0": "không", "1": "một", "2": "hai", "3": "ba", "4": "bốn", "5": "năm", "6": "sáu", "7": "bảy", "8": "tám", "9": "chín", "10": "mười", "90": "chín mươi"}
YANSHEE_ALIASES = ["yamsea", "yan see", "dân si", "gian si", "dân xe", "giang sê", "dàn xe", "giàn xe", "gian xe"]

def normalize_vi(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    for alias in YANSHEE_ALIASES:
        text = text.replace(alias, "yanshee")
    return " ".join([NUM_TO_VI.get(w, w) for w in text.split()])

def compute_corpus_metrics(refs, hyps):
    total_S, total_D, total_I, total_N = 0, 0, 0, 0
    total_char_err, total_char_N = 0, 0
    for r_str, h_str in zip(refs, hyps):
        r, h = normalize_vi(r_str).split(), normalize_vi(h_str).split()
        n, m = len(r), len(h)
        dp = [[(0, 0, 0, 0) for _ in range(m + 1)] for _ in range(n + 1)]
        for i in range(1, n + 1): dp[i][0] = (i, 0, i, 0)
        for j in range(1, m + 1): dp[0][j] = (j, 0, 0, j)
        for i in range(1, n + 1):
            for j in range(1, m + 1):
                if r[i - 1] == h[j - 1]: dp[i][j] = dp[i - 1][j - 1]
                else:
                    s_c, s_S, s_D, s_I = dp[i - 1][j - 1]
                    d_c, d_S, d_D, d_I = dp[i - 1][j]
                    i_c, i_S, i_D, i_I = dp[i][j - 1]
                    dp[i][j] = min([(s_c+1, s_S+1, s_D, s_I), (d_c+1, d_S, d_D+1, d_I), (i_c+1, i_S, i_D, i_I+1)], key=lambda x: x[0])
        _, S, D, I = dp[n][m]
        total_S += S; total_D += D; total_I += I; total_N += max(1, n)

        # Tính CER nhanh
        rc, hc = list(normalize_vi(r_str)), list(normalize_vi(h_str))
        nc, mc = len(rc), len(hc)
        dpc = [[0] * (mc + 1) for _ in range(nc + 1)]
        for i in range(nc + 1): dpc[i][0] = i
        for j in range(mc + 1): dpc[0][j] = j
        for i in range(1, nc + 1):
            for j in range(1, mc + 1):
                dpc[i][j] = min(dpc[i-1][j]+1, dpc[i][j-1]+1, dpc[i-1][j-1] + (0 if rc[i-1]==hc[j-1] else 1))
        total_char_err += dpc[nc][mc]
        total_char_N += max(1, nc)

    wer = ((total_S + total_D + total_I) / total_N) * 100.0
    cer = (total_char_err / total_char_N) * 100.0
    return wer, cer, (total_S/total_N)*100, (total_D/total_N)*100, (total_I/total_N)*100

# 3. ĐÁNH GIÁ SO SÁNH 4 MÔ HÌNH (Như Bảng 5 & Bảng 8 trong bài báo PDF)
MODELS_TO_EVAL = [
    ("1. Teacher (PhoWhisper-large, 32L Dec)", "vinai/PhoWhisper-large"),
    ("2. Baseline (PhoWhisper-base, 6L Dec)", "vinai/PhoWhisper-base"),
    ("3. Student Untrained (2L Dec - Chưa train)", STUDENT_INIT_DIR),
    ("4. Distil-PhoWhisper (2L Dec - Đã Distill)", STUDENT_TRAINED_DIR),
]

results = []
refs = [item["sentence"] for item in test_ds]
audios = [item["audio"]["array"] for item in test_ds]
input_feats = processor(audios, sampling_rate=16000, return_tensors="pt").input_features.to(device, dtype=amp_dtype)

for label_name, model_path in MODELS_TO_EVAL:
    if not os.path.exists(model_path) and not model_path.startswith("vinai/"):
        continue
    print(f"⏱️ Đang chấm điểm: {label_name} ...")
    m = WhisperForConditionalGeneration.from_pretrained(model_path, torch_dtype=amp_dtype).to(device)
    m.eval()
    params_m = sum(p.numel() for p in m.parameters()) / 1e6

    # Chạy theo lô (Batch = 10) để đo thời gian chính xác
    hyps = []
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for i in range(0, len(test_ds), 10):
            batch_inp = input_feats[i:i+10]
            gen_ids = m.generate(batch_inp, language="vi", task="transcribe", num_beams=1, max_new_tokens=45)
            hyps.extend(processor.batch_decode(gen_ids, skip_special_tokens=True))
    if device == "cuda": torch.cuda.synchronize()
    avg_latency_ms = ((time.time() - t0) / len(test_ds)) * 1000.0

    wer, cer, ser, der, ier = compute_corpus_metrics(refs, hyps)
    results.append({
        "Model": label_name,
        "Params (M)": round(params_m, 1),
        "Latency/câu (ms)": round(avg_latency_ms, 1),
        "WER (%)": round(wer, 2),
        "CER (%)": round(cer, 2),
        "SER (Thay thế %)": round(ser, 1),
        "DER (Thiếu %)": round(der, 1),
        "IER (Thừa/Lặp %)": round(ier, 1),
    })
    del m
    if device == "cuda": torch.cuda.empty_cache()

df_res = pd.DataFrame(results)
teacher_lat = df_res.iloc[0]["Latency/câu (ms)"]
df_res["Tăng tốc (Speed-up)"] = [f"{teacher_lat / r['Latency/câu (ms)']:.2f}x" for _, r in df_res.iterrows()]

print("\n" + "=" * 95)
print("📊 BẢNG KẾT QUẢ THỰC NGHIỆM (ĐỐI CHIẾU BẢNG 5 & BẢNG 8 BÀI BÁO DISTIL-WHISPER)")
print("=" * 95)
print(df_res.to_string(index=False))
df_res.to_csv(f"{BASE_DRIVE}/benchmark_results_ch8.csv", index=False)
print(f"\n💾 Đã lưu bảng kết quả vào: {BASE_DRIVE}/benchmark_results_ch8.csv")

📥 Đang chuẩn bị tập Test đánh giá (50 câu tiếng Việt thực tế)...
⏱️ Đang chấm điểm: 1. Teacher (PhoWhisper-large, 32L Dec) ...


Loading weights:   0%|          | 0/1260 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=45) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

⏱️ Đang chấm điểm: 2. Baseline (PhoWhisper-base, 6L Dec) ...


Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

...

[transformers] Both `max_new_tokens` (=45) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

⏱️ Đang chấm điểm: 4. Distil-PhoWhisper (2L Dec - Đã Distill) ...


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=45) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=45) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/


📊 BẢNG KẾT QUẢ THỰC NGHIỆM (ĐỐI CHIẾU BẢNG 5 & BẢNG 8 BÀI BÁO DISTIL-WHISPER)
                                     Model  Params (M)  Latency/câu (ms)  WER (%)  CER (%)  SER (Thay thế %)  DER (Thiếu %)  IER (Thừa/Lặp %) Tăng tốc (Speed-up)
    1. Teacher (PhoWhisper-large, 32L Dec)      1543.3             118.9     3.83     1.70               3.4            0.2               0.2               1.00x
     2. Baseline (PhoWhisper-base, 6L Dec)        72.6              27.4     6.81     2.50               6.6            0.2               0.0               4.34x
3. Student Untrained (2L Dec - Chưa train)        55.8              20.7   410.00   232.15              84.5           15.5             310.0               5.74x
4. Distil-PhoWhisper (2L Dec - Đã Distill)        55.8              14.7    12.13     5.44              10.0            1.1               1.1               8.09x

💾 Đã lưu bảng kết quả vào: /content/drive/MyDrive/Distil_Whisper_Vi_Yanshee/benchmark_results_ch8.csv


In [ ]:
import os
import time
import shutil
import psutil
import subprocess
import sys
import librosa

# 1. CÀI ĐẶT CTRANSLATE2 & FASTER-WHISPER
print("📦 [1/4] Đang kiểm tra CTranslate2 và faster-whisper...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ctranslate2", "faster-whisper", "psutil"])
import ctranslate2
from faster_whisper import WhisperModel
from transformers import AutoTokenizer, WhisperProcessor

BASE_DRIVE = "/content/drive/MyDrive/Distil_Whisper_Vi_Yanshee"
HF_TRAINED_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_trained"
CT2_INT8_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_ct2_int8"

# 2. ĐẢM BẢO CÓ ĐỦ FILE tokenizer.json & preprocessor_config.json TRONG THƯ MỤC MODEL
print("\n🔧 [2/4] Đang tạo file tokenizer.json chuẩn (Fast Tokenizer) & nén sang CTranslate2 INT8...")
# PhoWhisper-base sử dụng cùng bộ từ điển Multilingual Whisper với openai/whisper-base
fast_tokenizer = AutoTokenizer.from_pretrained("openai/whisper-base", use_fast=True)
fast_tokenizer.save_pretrained(HF_TRAINED_DIR)

processor = WhisperProcessor.from_pretrained("vinai/PhoWhisper-base")
processor.feature_extractor.save_pretrained(HF_TRAINED_DIR)

# Kiểm tra các file cần copy đã thực sự tồn tại chưa
copy_files = [f for f in ["tokenizer.json", "preprocessor_config.json"] if os.path.exists(os.path.join(HF_TRAINED_DIR, f))]
print(f"   -> Các file cấu hình đi kèm: {copy_files}")

# Gọi trực tiếp TransformersConverter bằng Python API (Rõ ràng, không bị lỗi ngầm của subprocess!)
converter = ctranslate2.converters.TransformersConverter(
    model_name_or_path=HF_TRAINED_DIR,
    copy_files=copy_files,
    load_as_float16=True
)
converter.convert(output_dir=CT2_INT8_DIR, quantization="int8", force=True)

# Tính dung lượng ổ cứng trước và sau khi nén int8
def get_dir_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024 ** 2)

size_hf_mb = get_dir_size_mb(HF_TRAINED_DIR)
size_int8_mb = get_dir_size_mb(CT2_INT8_DIR)
print(f"✅ NÉN INT8 THÀNH CÔNG! Đã lưu tại: {CT2_INT8_DIR}")
print(f"   - Dung lượng gốc (HuggingFace fp32)   : {size_hf_mb:.1f} MB")
print(f"   - Dung lượng sau nén (CTranslate2 int8): {size_int8_mb:.1f} MB (Giảm {(1 - size_int8_mb/size_hf_mb)*100:.1f}%!)")

# 3. ĐO DUNG LƯỢNG RAM & TỐC ĐỘ RTF TRÊN CPU (MÔ PHỎNG 4 NHÂN RASPBERRY PI 3B)
print("\n⏱️ [3/4] Đang nạp mô hình INT8 lên CPU (4 luồng - mô phỏng Quad-core Pi 3B) và đo RAM...")
process = psutil.Process(os.getpid())
ram_before_mb = process.memory_info().rss / (1024 ** 2)

ct2_model = WhisperModel(CT2_INT8_DIR, device="cpu", compute_type="int8", cpu_threads=4)

ram_after_mb = process.memory_info().rss / (1024 ** 2)
ram_used_mb = max(size_int8_mb, ram_after_mb - ram_before_mb)

# Lấy 1 file âm thanh câu lệnh Yanshee để test tốc độ trên CPU
RAW_WAV_DIR = f"{BASE_DRIVE}/data/raw_wavs"
test_wav = os.path.join(RAW_WAV_DIR, "sample_000.wav")
if not os.path.exists(test_wav):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gtts", "soundfile"])
    from gtts import gTTS
    import soundfile as sf
    tts = gTTS("Yanshee ơi hãy tiến lên phía trước hai bước rồi rẽ trái", lang="vi")
    tts.save("temp_pi3.mp3")
    y, _ = librosa.load("temp_pi3.mp3", sr=16000)
    test_wav = "temp_pi3.wav"
    sf.write(test_wav, y, 16000)

audio_data, _ = librosa.load(test_wav, sr=16000)
audio_duration_sec = len(audio_data) / 16000.0

# Chạy warm-up 1 lần trên CPU
segments, _ = ct2_model.transcribe(test_wav, language="vi", beam_size=1)
_ = list(segments)

# Đo chính xác thời gian suy luận trên CPU (trung bình 3 lần chạy)
times = []
pred_text = ""
for _ in range(3):
    t0 = time.time()
    segments, info = ct2_model.transcribe(
        test_wav,
        language="vi",
        beam_size=1,
        without_timestamps=True
    )
    pred_text = " ".join([seg.text for seg in segments]).strip()
    times.append(time.time() - t0)

avg_cpu_time_sec = sum(times) / len(times)
rtf_colab_cpu = avg_cpu_time_sec / audio_duration_sec
estimated_pi3_time_sec = avg_cpu_time_sec * 5.0
estimated_pi3_rtf = estimated_pi3_time_sec / audio_duration_sec

print("\n" + "=" * 80)
print("🤖 [4/4] BÁO CÁO TRIỂN KHAI INT8 CHO ROBOT YANSHEE (RASPBERRY PI 3B - 1GB RAM)")
print("=" * 80)
print(f"📌 Câu lệnh nhận dạng được (INT8 CPU) : '{pred_text}'")
print(f"📌 Độ dài file âm thanh               : {audio_duration_sec:.2f} giây")
print(f"💾 Kích thước Model INT8 trên thẻ nhớ : {size_int8_mb:.1f} MB")
print(f"🧠 Lượng RAM tăng thêm khi chạy       : ~{ram_used_mb:.1f} MB (An toàn tuyệt đối cho < 500MB RAM trống của Pi 3B!)")
print(f"⚡ Thời gian chạy trên CPU Colab (4c) : {avg_cpu_time_sec*1000:.1f} ms | RTF = {rtf_colab_cpu:.3f}")
print(f"🍓 Ước tính trên CPU Pi 3B (Cortex-A53): ~{estimated_pi3_time_sec:.2f} giây | RTF ước tính = {estimated_pi3_rtf:.2f}")
print("=" * 80)

📦 [1/4] Đang kiểm tra CTranslate2 và faster-whisper...

🔧 [2/4] Đang tạo file tokenizer.json chuẩn (Fast Tokenizer) & nén sang CTranslate2 INT8...


config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

   -> Các file cấu hình đi kèm: ['tokenizer.json', 'preprocessor_config.json']


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

✅ NÉN INT8 THÀNH CÔNG! Đã lưu tại: /content/drive/MyDrive/Distil_Whisper_Vi_Yanshee/models/distil_phowhisper_base_ct2_int8
   - Dung lượng gốc (HuggingFace fp32)   : 216.6 MB
   - Dung lượng sau nén (CTranslate2 int8): 59.3 MB (Giảm 72.6%!)

⏱️ [3/4] Đang nạp mô hình INT8 lên CPU (4 luồng - mô phỏng Quad-core Pi 3B) và đo RAM...


TypeError: open() got an unexpected keyword argument 'metadata_errors'

In [ ]:
import os
import time
import psutil
import librosa
import numpy as np
from faster_whisper import WhisperModel

BASE_DRIVE = "/content/drive/MyDrive/Distil_Whisper_Vi_Yanshee"
HF_TRAINED_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_trained"
CT2_INT8_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_ct2_int8"

def get_dir_size_mb(path):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, files in os.walk(path) for f in files) / (1024 ** 2)

size_hf_mb = get_dir_size_mb(HF_TRAINED_DIR)
size_int8_mb = get_dir_size_mb(CT2_INT8_DIR)

# Nạp mô hình INT8 lên CPU (4 nhân như Raspberry Pi 3B)
process = psutil.Process(os.getpid())
ram_before_mb = process.memory_info().rss / (1024 ** 2)
ct2_model = WhisperModel(CT2_INT8_DIR, device="cpu", compute_type="int8", cpu_threads=4)
ram_after_mb = process.memory_info().rss / (1024 ** 2)
ram_used_mb = max(size_int8_mb, ram_after_mb - ram_before_mb)

# Đọc mảng âm thanh 16kHz (float32) để truyền trực tiếp vào transcribe (không qua PyAV!)
RAW_WAV_DIR = f"{BASE_DRIVE}/data/raw_wavs"
test_wav = os.path.join(RAW_WAV_DIR, "sample_000.wav")
audio_data, _ = librosa.load(test_wav, sr=16000)
audio_data = audio_data.astype(np.float32)
audio_duration_sec = len(audio_data) / 16000.0

# Chạy warm-up 1 lần trên CPU với mảng numpy audio_data
segments, _ = ct2_model.transcribe(audio_data, language="vi", beam_size=1)
_ = list(segments)

# Đo chính xác thời gian suy luận trên CPU (trung bình 3 lần chạy)
times = []
pred_text = ""
for _ in range(3):
    t0 = time.time()
    segments, info = ct2_model.transcribe(
        audio_data,             # Truyền trực tiếp mảng sóng âm 16kHz!
        language="vi",
        beam_size=1,
        without_timestamps=True
    )
    pred_text = " ".join([seg.text for seg in segments]).strip()
    times.append(time.time() - t0)

avg_cpu_time_sec = sum(times) / len(times)
rtf_colab_cpu = avg_cpu_time_sec / audio_duration_sec
estimated_pi3_time_sec = avg_cpu_time_sec * 5.0
estimated_pi3_rtf = estimated_pi3_time_sec / audio_duration_sec

print("=" * 80)
print("🤖 [4/4] BÁO CÁO TRIỂN KHAI INT8 CHO ROBOT YANSHEE (RASPBERRY PI 3B - 1GB RAM)")
print("=" * 80)
print(f"📌 Câu lệnh nhận dạng được (INT8 CPU) : '{pred_text}'")
print(f"📌 Độ dài file âm thanh               : {audio_duration_sec:.2f} giây")
print(f"💾 Kích thước gốc (HuggingFace fp32)  : {size_hf_mb:.1f} MB")
print(f"💾 Kích thước sau nén (CT2 INT8)      : {size_int8_mb:.1f} MB (Giảm {(1 - size_int8_mb/size_hf_mb)*100:.1f}%!)")
print(f"🧠 Lượng RAM tăng thêm khi chạy       : ~{ram_used_mb:.1f} MB (An toàn tuyệt đối cho < 500MB RAM trống của Pi 3B!)")
print(f"⚡ Thời gian chạy trên CPU Colab (4c) : {avg_cpu_time_sec*1000:.1f} ms | RTF = {rtf_colab_cpu:.3f}")
print(f"🍓 Ước tính trên CPU Pi 3B (Cortex-A53): ~{estimated_pi3_time_sec:.2f} giây | RTF ước tính = {estimated_pi3_rtf:.2f}")
print("=" * 80)

🤖 [4/4] BÁO CÁO TRIỂN KHAI INT8 CHO ROBOT YANSHEE (RASPBERRY PI 3B - 1GB RAM)
📌 Câu lệnh nhận dạng được (INT8 CPU) : 'Yanshee ơi tiếng lên phía trước hai bước.'
📌 Độ dài file âm thanh               : 3.05 giây
💾 Kích thước gốc (HuggingFace fp32)  : 216.6 MB
💾 Kích thước sau nén (CT2 INT8)      : 59.3 MB (Giảm 72.6%!)
🧠 Lượng RAM tăng thêm khi chạy       : ~135.5 MB (An toàn tuyệt đối cho < 500MB RAM trống của Pi 3B!)
⚡ Thời gian chạy trên CPU Colab (4c) : 471.8 ms | RTF = 0.155
🍓 Ước tính trên CPU Pi 3B (Cortex-A53): ~2.36 giây | RTF ước tính = 0.77


In [ ]:
import time
import re
import unicodedata
import numpy as np
import pandas as pd
from datasets import load_dataset, Audio as HFAudio
from faster_whisper import WhisperModel
from IPython.display import Audio, display

BASE_DRIVE = "/content/drive/MyDrive/Distil_Whisper_Vi_Yanshee"
CT2_INT8_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_ct2_int8"

# 1. TẢI BỘ DỮ LIỆU MỚI HOÀN TOÀN (OOD): VIET-BUD500 (Tập Test)
OOD_REPO = "jbdc1/viet_bud500-asr-preprocessed"
NUM_TEST_OOD = 30
print(f"📥 [1/3] Đang tải {NUM_TEST_OOD} mẫu Test từ bộ dữ liệu mới tinh (OOD): '{OOD_REPO}'...")
ood_test_ds = load_dataset(OOD_REPO, split=f"test[:{NUM_TEST_OOD}]")
ood_test_ds = ood_test_ds.cast_column("audio", HFAudio(sampling_rate=16000))
print(f"✅ Đã tải xong {len(ood_test_ds)} mẫu OOD từ Viet-Bud500!")

# 2. NẠP MODEL NHỎ ĐÃ NÉN INT8 (59.3 MB) CHẠY TRÊN 4 NHÂN CPU
print(f"\n🧠 [2/3] Đang nạp Model nhỏ INT8 (59.3 MB) trên CPU (4 threads)...")
ct2_model = WhisperModel(CT2_INT8_DIR, device="cpu", compute_type="int8", cpu_threads=4)

# Hàm chuẩn hóa & tính WER / CER
def normalize_vi(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def calc_edit_rate(ref_list, hyp_list):
    n, m = len(ref_list), len(hyp_list)
    if n == 0: return 0.0
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1] + (0 if ref_list[i-1]==hyp_list[j-1] else 1))
    return dp[n][m], n

# 3. CHẠY TEST TRÊN TOÀN BỘ 30 CÂU VIET-BUD500 & IN CHI TIẾT
print(f"\n🚀 [3/3] Đang chạy Model INT8 (59.3 MB) trên {NUM_TEST_OOD} câu Viet-Bud500...\n")
print("=" * 95)

total_w_err, total_w_len = 0, 0
total_c_err, total_c_len = 0, 0
total_audio_sec, total_infer_sec = 0.0, 0.0

for idx, item in enumerate(ood_test_ds):
    audio_arr = item["audio"]["array"].astype(np.float32)
    dur_sec = len(audio_arr) / 16000.0
    gt_text = item["transcription"]
    domain = item.get("domain", "general")

    t0 = time.time()
    segments, _ = ct2_model.transcribe(
        audio_arr,
        language="vi",
        beam_size=1,
        without_timestamps=True
    )
    pred_text = " ".join([s.text for s in segments]).strip()
    infer_sec = time.time() - t0

    norm_gt = normalize_vi(gt_text)
    norm_pred = normalize_vi(pred_text)

    w_err, w_len = calc_edit_rate(norm_gt.split(), norm_pred.split())
    c_err, c_len = calc_edit_rate(list(norm_gt), list(norm_pred))

    total_w_err += w_err; total_w_len += w_len
    total_c_err += c_err; total_c_len += c_len
    total_audio_sec += dur_sec; total_infer_sec += infer_sec

    sample_wer = (w_err / max(1, w_len)) * 100.0
    # In ra 10 câu đầu tiên để bạn soi từng chữ!
    if idx < 10:
        print(f"Mẫu #{idx+1:02d} [{domain}] (Dài {dur_sec:.1f}s | Giải mã CPU: {infer_sec*1000:.0f}ms | WER câu = {sample_wer:.1f}%)")
        print(f"   📌 Gốc (Bud500)      : '{gt_text}'")
        print(f"   🤖 Model INT8 (59MB) : '{pred_text}'\n")

overall_wer = (total_w_err / total_w_len) * 100.0
overall_cer = (total_c_err / total_c_len) * 100.0
overall_rtf = total_infer_sec / total_audio_sec

print("=" * 95)
print(f"📊 TỔNG KẾT ĐÁNH GIÁ OOD TRÊN BỘ DỮ LIỆU MỚI 'VIET-BUD500' ({NUM_TEST_OOD} MẪU):")
print(f"   - Model kiểm tra            : Distil-PhoWhisper-base (6 Enc / 2 Dec) - Nén INT8 (59.3 MB)")
print(f"   - Thiết bị chạy             : Thuần CPU (4 threads - không dùng GPU)")
print(f"   - Tổng thời lượng âm thanh  : {total_audio_sec:.1f} giây")
print(f"   - Tổng thời gian CPU giải mã: {total_infer_sec:.2f} giây -> RTF = {overall_rtf:.3f} (Nhanh gấp {1/overall_rtf:.1f} lần thời gian thực!)")
print(f"   - Độ lỗi từ OOD (WER %)     : {overall_wer:.2f}%")
print(f"   - Độ lỗi ký tự OOD (CER %)  : {overall_cer:.2f}%")
print("=" * 95)

# Nghe thử mẫu số 1 của Viet-Bud500
print("\n🎧 Nghe thử âm thanh Mẫu #01 của Viet-Bud500:")
display(Audio(ood_test_ds[0]["audio"]["array"], rate=16000))

📥 [1/3] Đang tải 30 mẫu Test từ bộ dữ liệu mới tinh (OOD): 'jbdc1/viet_bud500-asr-preprocessed'...


README.md:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/65 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/65 [00:00<?, ?it/s]

data/train-00000-of-00065.parquet: reconstructing file:   0%|          |  0.00B /  495MB            

...

data/test-00007-of-00008.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/389106 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/48638 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/48639 [00:00<?, ? examples/s]

✅ Đã tải xong 30 mẫu OOD từ Viet-Bud500!

🧠 [2/3] Đang nạp Model nhỏ INT8 (59.3 MB) trên CPU (4 threads)...

🚀 [3/3] Đang chạy Model INT8 (59.3 MB) trên 30 câu Viet-Bud500...

Mẫu #01 [viet_bud500] (Dài 2.6s | Giải mã CPU: 463ms | WER câu = 22.2%)
   📌 Gốc (Bud500)      : 'cũng mãi mãi phải ở bên nhau lúc này'
   🤖 Model INT8 (59MB) : 'ũng mãi phải ở bên nhau lúc này.'

Mẫu #02 [viet_bud500] (Dài 3.9s | Giải mã CPU: 476ms | WER câu = 9.5%)
   📌 Gốc (Bud500)      : 'năng lượng được phước từ khán giả thôi cho nên là nó mới có được cái cảm giác thăng hoa đấy'
   🤖 Model INT8 (59MB) : 'năng lượng được phước từ khán giả thôi cho nên là nó mới có được cái cảm tắc thăng hoa để'

Mẫu #03 [viet_bud500] (Dài 2.8s | Giải mã CPU: 460ms | WER câu = 20.0%)
   📌 Gốc (Bud500)      : 'sẽ tìm ra thủ phạm và mình được thả về'
   🤖 Model INT8 (59MB) : 'sẽ tìm ra thú phạm và mình được thá về.'

Mẫu #04 [viet_bud500] (Dài 2.9s | Giải mã CPU: 452ms | WER câu = 11.1%)
   📌 Gốc (Bud500)      : 'ở khoảng cách g

In [ ]:
import os
import time
import re
import shutil
import unicodedata
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset, load_from_disk, concatenate_datasets, Audio as HFAudio
from transformers import WhisperForConditionalGeneration, WhisperProcessor, AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
import ctranslate2
from faster_whisper import WhisperModel

device = "cuda" if torch.cuda.is_available() else "cpu"
amp_dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print(f"🖥️ Thiết bị: {device.upper()} ({torch.cuda.get_device_name(0)}) | Precision: {amp_dtype}")

BASE_DRIVE = "/content/drive/MyDrive/Distil_Whisper_Vi_Yanshee"
BEST_MODEL_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_trained"
CT2_INT8_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_ct2_int8"
CHECKPOINT_DIR = f"{BASE_DRIVE}/checkpoints_large_run"

# =====================================================================
# 1. NẠP DỮ LIỆU ĐA MIỀN LỚN: 4,000 VIVOS-FLEURS MỚI + 4,000 VIET-BUD500 + YANSHEE
# =====================================================================
print("\n📥 [1/5] Đang tải thêm dữ liệu đa miền (VIVOS-FLEURS mới + Viet-Bud500 đời thực)...")

# Lấy 4,000 mẫu tiếp theo của VIVOS-FLEURS (từ 2000 -> 6000, chưa từng học ở đợt 1)
ds_vivos = load_dataset("JRHuy/vivos-fleurs", split="train[2000:6000]")
col_v = "raw_transcription" if "raw_transcription" in ds_vivos.column_names else "transcription"
ds_vivos = ds_vivos.rename_column(col_v, "sentence").cast_column("audio", HFAudio(sampling_rate=16000))
ds_vivos = ds_vivos.remove_columns([c for c in ds_vivos.column_names if c not in ["audio", "sentence"]])

# Lấy 4,000 mẫu đời thực từ tập TRAIN của Viet-Bud500
ds_bud = load_dataset("jbdc1/viet_bud500-asr-preprocessed", split="train[:4000]")
ds_bud = ds_bud.rename_column("transcription", "sentence").cast_column("audio", HFAudio(sampling_rate=16000))
ds_bud = ds_bud.remove_columns([c for c in ds_bud.column_names if c not in ["audio", "sentence"]])

# Trộn thêm tập câu lệnh Robot Yanshee
PSEUDO_YANSHEE_DIR = f"{BASE_DRIVE}/data/yanshee_vi_pseudo_labelled"
if os.path.exists(PSEUDO_YANSHEE_DIR):
    y_ds = load_from_disk(PSEUDO_YANSHEE_DIR)["train"]
    y_ds = y_ds.remove_columns([c for c in y_ds.column_names if c not in ["audio", "whisper_transcript"]])
    y_ds = y_ds.rename_column("whisper_transcript", "sentence").cast_column("audio", HFAudio(sampling_rate=16000))
    multi_train = concatenate_datasets([ds_vivos, ds_bud] + [y_ds] * 25).shuffle(seed=100)
else:
    multi_train = concatenate_datasets([ds_vivos, ds_bud]).shuffle(seed=100)

print(f"✅ Đã gộp xong bộ dữ liệu Đa Miền: Tổng cộng {len(multi_train)} mẫu huấn luyện!")

# =====================================================================
# 2. NẠP TEACHER & NẠP STUDENT ĐÃ TRAIN Ở ĐỢT 1 ĐỂ HỌC NỐI TIẾP!
# =====================================================================
TEACHER_ID = "vinai/PhoWhisper-large"
print(f"\n🎓 [2/5] Đang nạp Teacher ({TEACHER_ID}) và Student từ Đợt 1 để học tiếp...")

processor = WhisperProcessor.from_pretrained("vinai/PhoWhisper-base")
processor.tokenizer.set_prefix_tokens(language="vi", task="transcribe", predict_timestamps=False)

teacher_model = WhisperForConditionalGeneration.from_pretrained(TEACHER_ID, torch_dtype=amp_dtype).to(device)
teacher_model.eval()

# Nạp thẳng bản Student đã train ở đợt 1 (không khởi tạo lại từ đầu!)
student_model = WhisperForConditionalGeneration.from_pretrained(BEST_MODEL_DIR).to(device)
student_model.freeze_encoder()  # Tiếp tục khóa cứng Encoder

# =====================================================================
# 3. HUẤN LUYỆN TIẾP TRÊN A100 VỚI BATCH_SIZE = 32 (TẬN DỤNG 40GB VRAM)
# =====================================================================
BATCH_SIZE = 32   # A100 40GB chạy Batch 32 cực nhanh!
EPOCHS = 3
bos_id = student_model.config.decoder_start_token_id

def collate_fn(batch):
    audios = [item["audio"]["array"] for item in batch]
    input_features = processor(audios, sampling_rate=16000, return_tensors="pt").input_features
    texts = [str(item["sentence"]).lower() if "Yanshee" not in str(item["sentence"]) else str(item["sentence"]) for item in batch]

    labels_batch = processor.tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    labels = labels_batch.input_ids.masked_fill(labels_batch.attention_mask.ne(1), -100)
    if (labels[:, 0] == bos_id).all().cpu().item():
        labels = labels[:, 1:]
    return {"input_features": input_features, "labels": labels}

train_loader = DataLoader(multi_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

TEMPERATURE = 2.0
def compute_kd_loss(student_logits, teacher_logits, labels):
    vocab_size = student_logits.size(-1)
    loss_pl = F.cross_entropy(student_logits.view(-1, vocab_size), labels.view(-1), ignore_index=-100)
    valid_mask = (labels != -100).unsqueeze(-1)
    s_log_probs = F.log_softmax(student_logits.float() / TEMPERATURE, dim=-1)
    t_probs = F.softmax(teacher_logits.float() / TEMPERATURE, dim=-1)
    kl_token = F.kl_div(s_log_probs, t_probs, reduction="none") * (TEMPERATURE ** 2)
    loss_kl = (kl_token * valid_mask).sum() / valid_mask.sum().clamp(min=1.0)
    return 0.8 * loss_kl + 1.0 * loss_pl, loss_kl.item(), loss_pl.item()

total_steps = len(train_loader) * EPOCHS
# Vì học nối tiếp (Fine-tuning tiếp), ta dùng LR = 1e-4 (nhẹ hơn 2e-4 ở đợt 1 để hội tụ sâu)
optimizer = torch.optim.AdamW([p for p in student_model.parameters() if p.requires_grad], lr=1e-4, betas=(0.9, 0.999))
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=max(15, int(0.05 * total_steps)), num_training_steps=total_steps)

print(f"\n🚀 [3/5] BẮT ĐẦU HỌC TIẾP ĐỢT 2 ({len(multi_train)} mẫu | Batch={BATCH_SIZE} | {EPOCHS} Epochs = {total_steps} Steps)...")
for epoch in range(1, EPOCHS + 1):
    student_model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Round-2 Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        inp = batch["input_features"].to(device)
        lbl = batch["labels"].to(device)

        optimizer.zero_grad()
        with torch.no_grad():
            t_logits = teacher_model(input_features=inp.to(amp_dtype), labels=lbl).logits
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
            s_logits = student_model(input_features=inp, labels=lbl).logits
            loss, l_kl, l_pl = compute_kd_loss(s_logits, t_logits, lbl)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix({"L_KD": f"{loss.item():.2f}", "KL": f"{l_kl:.2f}", "PL": f"{l_pl:.2f}"})

    # Lưu ngay sau mỗi Epoch
    student_model.save_pretrained(BEST_MODEL_DIR)
    print(f"   -> Kết thúc Epoch {epoch:02d}/{EPOCHS:02d} | Loss trung bình: {total_loss/len(train_loader):.4f} (Đã lưu cập nhật vào Drive!)")

# Giải phóng VRAM GPU
del teacher_model, student_model
torch.cuda.empty_cache()

# =====================================================================
# 4. TỰ ĐỘNG NÉN LẠI SANG CTRANSLATE2 INT8 (59.3 MB) & CẬP NHẬT FILE ZIP
# =====================================================================
print("\n🗜️ [4/5] Đang nén bản Model V2 vừa học xong sang CTranslate2 INT8 (59.3 MB)...")
fast_tokenizer = AutoTokenizer.from_pretrained("openai/whisper-base", use_fast=True)
fast_tokenizer.save_pretrained(BEST_MODEL_DIR)
processor.feature_extractor.save_pretrained(BEST_MODEL_DIR)

converter = ctranslate2.converters.TransformersConverter(
    model_name_or_path=BEST_MODEL_DIR,
    copy_files=["tokenizer.json", "preprocessor_config.json"],
    load_as_float16=True
)
converter.convert(output_dir=CT2_INT8_DIR, quantization="int8", force=True)

# Cập nhật luôn file ZIP gửi cho bạn của bạn
zip_out = f"{BASE_DRIVE}/Yanshee_Distil_PhoWhisper_INT8_Package"
shutil.make_archive(zip_out, "zip", CT2_INT8_DIR)
print(f"✅ Đã nén xong INT8 V2 và cập nhật gói ZIP tại: {zip_out}.zip")

# =====================================================================
# 5. ĐO LẠI NGAY TRÊN 30 CÂU TEST VIET-BUD500 ĐỂ SO SÁNH VỚI ĐỢT 1 (21.86%)!
# =====================================================================
print("\n📊 [5/5] Đang kiểm tra lại Model INT8 V2 trên 30 câu Test Viet-Bud500 (trên CPU)...")
ood_test_ds = load_dataset("jbdc1/viet_bud500-asr-preprocessed", split="test[:30]").cast_column("audio", HFAudio(sampling_rate=16000))
ct2_model = WhisperModel(CT2_INT8_DIR, device="cpu", compute_type="int8", cpu_threads=4)

def normalize_vi(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def calc_edit_rate(ref_list, hyp_list):
    n, m = len(ref_list), len(hyp_list)
    if n == 0: return 0, 0
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1] + (0 if ref_list[i-1]==hyp_list[j-1] else 1))
    return dp[n][m], n

total_w_err, total_w_len, total_c_err, total_c_len = 0, 0, 0, 0
for idx, item in enumerate(ood_test_ds):
    audio_arr = item["audio"]["array"].astype(np.float32)
    segments, _ = ct2_model.transcribe(audio_arr, language="vi", beam_size=1, without_timestamps=True)
    pred_text = " ".join([s.text for s in segments]).strip()

    norm_gt, norm_pred = normalize_vi(item["transcription"]), normalize_vi(pred_text)
    we, wl = calc_edit_rate(norm_gt.split(), norm_pred.split())
    ce, cl = calc_edit_rate(list(norm_gt), list(norm_pred))
    total_w_err += we; total_w_len += wl; total_c_err += ce; total_c_len += cl
    if idx < 5:
        print(f"   Mẫu #{idx+1:02d} | Gốc: '{item['transcription']}' -> INT8 V2: '{pred_text}'")

new_wer = (total_w_err / total_w_len) * 100.0
new_cer = (total_c_err / total_c_len) * 100.0

print("\n" + "=" * 85)
print("🏆 SO SÁNH KẾT QUẢ TRÊN VIET-BUD500 TRƯỚC VÀ SAU KHI HỌC TIẾP ĐỢT 2 (8,200 MẪU):")
print(f"   - Đợt 1 (Chỉ học 2,000 câu VIVOS)          : WER = 21.86% | CER = 10.79%")
print(f"   - Đợt 2 (Học tiếp +8,200 câu Đa Miền V2)   : WER = {new_wer:.2f}% | CER = {new_cer:.2f}% (Dung lượng vẫn 59.3 MB!)")
print("=" * 85)

🖥️ Thiết bị: CUDA (NVIDIA A100-SXM4-40GB) | Precision: torch.bfloat16

📥 [1/5] Đang tải thêm dữ liệu đa miền (VIVOS-FLEURS mới + Viet-Bud500 đời thực)...


Resolving data files:   0%|          | 0/65 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/65 [00:00<?, ?it/s]

✅ Đã gộp xong bộ dữ liệu Đa Miền: Tổng cộng 8200 mẫu huấn luyện!

🎓 [2/5] Đang nạp Teacher (vinai/PhoWhisper-large) và Student từ Đợt 1 để học tiếp...


Loading weights:   0%|          | 0/1260 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

✅ Đã nén xong INT8 V2 và cập nhật gói ZIP tại: /content/drive/MyDrive/Distil_Whisper_Vi_Yanshee/Yanshee_Distil_PhoWhisper_INT8_Package.zip

📊 [5/5] Đang kiểm tra lại Model INT8 V2 trên 30 câu Test Viet-Bud500 (trên CPU)...


Resolving data files:   0%|          | 0/65 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/65 [00:00<?, ?it/s]

   Mẫu #01 | Gốc: 'cũng mãi mãi phải ở bên nhau lúc này' -> INT8 V2: 'cũng mãi phải ở bên nhau lúc này.'
   Mẫu #02 | Gốc: 'năng lượng được phước từ khán giả thôi cho nên là nó mới có được cái cảm giác thăng hoa đấy' -> INT8 V2: 'năng lượng được phước từ khán giả thôi cho nên là nó mới có được cái cảm giác thăng hoa để.'
   Mẫu #03 | Gốc: 'sẽ tìm ra thủ phạm và mình được thả về' -> INT8 V2: 'sẽ tìm ra thú phạm và mình được thá về.'
   Mẫu #04 | Gốc: 'ở khoảng cách gần thư kỳ không tài nào' -> INT8 V2: 'ở khoảng cách gần thư kỳ không tải nào.'
   Mẫu #05 | Gốc: 'mẹ vẫn túc tắc xoa xoa lên bụng chị dâu' -> INT8 V2: 'mẹ vẫn túc tắt xoa lên bụng chị dốc.'

🏆 SO SÁNH KẾT QUẢ TRÊN VIET-BUD500 TRƯỚC VÀ SAU KHI HỌC TIẾP ĐỢT 2 (8,200 MẪU):
   - Đợt 1 (Chỉ học 2,000 câu VIVOS)          : WER = 21.86% | CER = 10.79%
   - Đợt 2 (Học tiếp +8,200 câu Đa Miền V2)   : WER = 11.47% | CER = 5.35% (Dung lượng vẫn 59.3 MB!)


In [ ]:
import os
import shutil
import librosa
import numpy as np
from faster_whisper import WhisperModel

BASE_DRIVE = "/content/drive/MyDrive/Distil_Whisper_Vi_Yanshee"
CT2_INT8_DIR = f"{BASE_DRIVE}/models/distil_phowhisper_base_ct2_int8"

# 1. ĐÓNG GÓI LẠI FILE ZIP CÓ SẴN THƯ MỤC CON 'distil_phowhisper_int8' BÊN TRONG
# (Để khi bạn của bạn giải nén ra là có sẵn đúng thư mục './distil_phowhisper_int8' không bao giờ lệch đường dẫn!)
PACKAGE_ROOT = "/content/package_for_friend"
SUBFOLDER = os.path.join(PACKAGE_ROOT, "distil_phowhisper_int8")
if os.path.exists(PACKAGE_ROOT):
    shutil.rmtree(PACKAGE_ROOT)
shutil.copytree(CT2_INT8_DIR, SUBFOLDER)

zip_out = f"{BASE_DRIVE}/Yanshee_Distil_PhoWhisper_INT8_Package"
shutil.make_archive(zip_out, "zip", PACKAGE_ROOT)
print(f"🎁 Đã cập nhật file ZIP chuẩn tại: {zip_out}.zip")

# 2. MÔ PHỎNG ĐÚNG NHƯ MÁY CỦA BẠN BẠN: Giải nén file ZIP ra '/content/test_unzip' và chạy thử!
TEST_UNZIP_DIR = "/content/test_unzip"
shutil.unpack_archive(f"{zip_out}.zip", TEST_UNZIP_DIR)

model_path = os.path.join(TEST_UNZIP_DIR, "distil_phowhisper_int8")
print(f"📂 Đang nạp model từ thư mục vừa giải nén: {model_path}")
model = WhisperModel(model_path, device="cpu", compute_type="int8", cpu_threads=4)

# Lấy 1 file wav bất kỳ để test
test_wav = f"{BASE_DRIVE}/data/raw_wavs/sample_000.wav"
audio_16k, _ = librosa.load(test_wav, sr=16000)
segments, _ = model.transcribe(audio_16k.astype(np.float32), language="vi", beam_size=1, without_timestamps=True)

print("🤖 Robot nghe được:", " ".join([s.text for s in segments]).strip())

🎁 Đã cập nhật file ZIP chuẩn tại: /content/drive/MyDrive/Distil_Whisper_Vi_Yanshee/Yanshee_Distil_PhoWhisper_INT8_Package.zip
📂 Đang nạp model từ thư mục vừa giải nén: /content/test_unzip/distil_phowhisper_int8
🤖 Robot nghe được: Yanse ơi tiếng lên phía trước hai bước.
